In [ ]:
from __future__ import annotations

from dataclasses import dataclass

from pathlib import Path

from typing import Dict, List, Optional



import pandas as pd





@dataclass

class WindowSplit:

    split_id: int

    train_start: pd.Timestamp

    train_end: pd.Timestamp

    val_start: pd.Timestamp

    val_end: pd.Timestamp

    train_df: pd.DataFrame

    val_df: pd.DataFrame





class DayLevelDatasetBuilder:

    def __init__(

        self,

        reports_dir: str = "reports_txt_by_day",

        returns_file: str = "all_stock_returns.csv",

        mapping_file: str = "Eastmoney_report_pdf_download/HS300.csv",

        min_text_length: int = 50,

        horizon_months: int = 3,

    ) -> None:

        self.reports_dir = Path(reports_dir)

        self.returns_file = Path(returns_file)

        self.mapping_file = Path(mapping_file)

        self.min_text_length = min_text_length

        self.horizon_months = horizon_months



        if not self.returns_file.exists():

            alt = Path("all_stock_return.csv")

            if alt.exists():

                self.returns_file = alt



        self.name_to_code = self._load_name_to_code()

        self.returns_df = self._load_returns()



    def _load_name_to_code(self) -> Dict[str, str]:

        mapping = pd.read_csv(self.mapping_file, dtype={"股票代码": str})

        mapping["股票代码"] = mapping["股票代码"].str.zfill(6)

        return dict(zip(mapping["股票简称"], mapping["股票代码"]))



    def _load_returns(self) -> pd.DataFrame:

        df = pd.read_csv(self.returns_file)

        df["交易日期"] = pd.to_datetime(df["交易日期"], errors="coerce")

        df = df.dropna(subset=["交易日期"]).sort_values("交易日期")



        for col in df.columns:

            if col == "交易日期":

                continue

            df[col] = pd.to_numeric(df[col], errors="coerce")

        return df



    @staticmethod

    def _read_text(txt_path: Path) -> Optional[str]:

        for enc in ("utf-8", "gbk", "gb18030"):

            try:

                return txt_path.read_text(encoding=enc).strip()

            except Exception:

                continue

        return None



    def _resolve_stock_code(self, company_name: str) -> Optional[str]:

        if company_name in self.name_to_code:

            return self.name_to_code[company_name]



        if company_name.isdigit():

            code = company_name.zfill(6)

            if code in self.returns_df.columns:

                return code



        return None



    def _calc_forward_return(self, stock_code: str, report_date: pd.Timestamp) -> Optional[float]:

        if stock_code not in self.returns_df.columns:

            return None



        end_date = report_date + pd.DateOffset(months=self.horizon_months)

        mask = (self.returns_df["交易日期"] >= report_date) & (self.returns_df["交易日期"] <= end_date)

        series = self.returns_df.loc[mask, stock_code].dropna()

        if series.empty:

            return None



        cumulative_return = (1.0 + series / 100.0).prod() - 1.0

        return float(cumulative_return)



    def build_dataset(self) -> pd.DataFrame:

        rows: List[Dict] = []



        for day_dir in sorted(self.reports_dir.iterdir()):

            if not day_dir.is_dir() or not day_dir.name.isdigit() or len(day_dir.name) != 8:

                continue



            report_date = pd.to_datetime(day_dir.name, format="%Y%m%d", errors="coerce")

            if pd.isna(report_date):

                continue



            for company_dir in day_dir.iterdir():

                if not company_dir.is_dir():

                    continue



                company_name = company_dir.name

                stock_code = self._resolve_stock_code(company_name)

                if stock_code is None:

                    continue



                label = self._calc_forward_return(stock_code, report_date)

                if label is None:

                    continue



                for txt_path in company_dir.rglob("*.txt"):

                    text = self._read_text(txt_path)

                    if not text or len(text) < self.min_text_length:

                        continue



                    rows.append(

                        {

                            "text": text,

                            "label": label,

                            "report_date": report_date.strftime("%Y-%m-%d"),

                            "stock_code": stock_code,

                            "company_name": company_name,

                            "horizon_months": self.horizon_months,

                            "source_file": txt_path.as_posix(),

                        }

                    )



        dataset = pd.DataFrame(rows)

        if dataset.empty:

            return dataset



        dataset["report_date"] = pd.to_datetime(dataset["report_date"])

        dataset = dataset.sort_values(["report_date", "stock_code"]).reset_index(drop=True)

        return dataset





def build_rolling_windows(

    dataset: pd.DataFrame,

    train_months: int = 24,

    val_months: int = 3,

    step_months: int = 1,

) -> List[WindowSplit]:

    if dataset.empty:

        return []



    df = dataset.copy()

    df["report_date"] = pd.to_datetime(df["report_date"])

    df = df.sort_values("report_date")



    min_date = df["report_date"].min().normalize()

    max_date = df["report_date"].max().normalize()



    splits: List[WindowSplit] = []

    split_id = 0



    val_start = min_date + pd.DateOffset(months=train_months)

    while val_start < max_date:

        train_start = val_start - pd.DateOffset(months=train_months)

        train_end = val_start

        val_end = val_start + pd.DateOffset(months=val_months)



        train_mask = (df["report_date"] >= train_start) & (df["report_date"] < train_end)

        val_mask = (df["report_date"] >= val_start) & (df["report_date"] < val_end)



        train_df = df.loc[train_mask].reset_index(drop=True)

        val_df = df.loc[val_mask].reset_index(drop=True)



        if not train_df.empty and not val_df.empty:

            split_id += 1

            splits.append(

                WindowSplit(

                    split_id=split_id,

                    train_start=train_start,

                    train_end=train_end,

                    val_start=val_start,

                    val_end=val_end,

                    train_df=train_df,

                    val_df=val_df,

                )

            )



        val_start = val_start + pd.DateOffset(months=step_months)



    return splits





def save_rolling_windows(splits: List[WindowSplit], output_dir: str = "rolling_windows") -> None:

    out_dir = Path(output_dir)

    out_dir.mkdir(parents=True, exist_ok=True)



    meta_rows: List[Dict] = []

    for split in splits:

        train_path = out_dir / f"train_split_{split.split_id:03d}.csv"

        val_path = out_dir / f"val_split_{split.split_id:03d}.csv"

        split.train_df.to_csv(train_path, index=False, encoding="utf-8-sig")

        split.val_df.to_csv(val_path, index=False, encoding="utf-8-sig")



        meta_rows.append(

            {

                "split_id": split.split_id,

                "train_start": split.train_start.strftime("%Y-%m-%d"),

                "train_end_exclusive": split.train_end.strftime("%Y-%m-%d"),

                "val_start": split.val_start.strftime("%Y-%m-%d"),

                "val_end_exclusive": split.val_end.strftime("%Y-%m-%d"),

                "train_size": len(split.train_df),

                "val_size": len(split.val_df),

                "train_file": train_path.as_posix(),

                "val_file": val_path.as_posix(),

            }

        )



    pd.DataFrame(meta_rows).to_csv(out_dir / "rolling_splits_meta.csv", index=False, encoding="utf-8-sig")

    print(f"Rolling splits saved to: {out_dir}")


未找到映射: 比亚迪电子
未找到映射: 中航电测
未找到映射: 中航电测
生成完成: train_5class.csv 和 val_5class.csv


In [3]:
# ===== 配置参数 =====

REPORTS_DIR = "reports_txt_by_day"

RETURNS_FILE = "all_stock_returns.csv"   # 若不存在，代码会自动尝试 all_stock_return.csv

MAPPING_FILE = "Eastmoney_report_pdf_download/HS300.csv"

OUTPUT_FILE = "daily_text_3m_return_dataset.csv"



HORIZON_MONTHS = 3

MIN_TEXT_LENGTH = 50



# rolling window 参数

BUILD_ROLLING = True

TRAIN_MONTHS = 24

VAL_MONTHS = 3

STEP_MONTHS = 1

ROLLING_OUTPUT_DIR = "rolling_windows"



# ===== 构建日度数据集 =====

builder = DayLevelDatasetBuilder(

    reports_dir=REPORTS_DIR,

    returns_file=RETURNS_FILE,

    mapping_file=MAPPING_FILE,

    min_text_length=MIN_TEXT_LENGTH,

    horizon_months=HORIZON_MONTHS,

)



dataset = builder.build_dataset()

if dataset.empty:

    raise ValueError("No valid samples generated. Please check directory and data coverage.")



dataset.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"Dataset saved: {OUTPUT_FILE}, samples={len(dataset)}")

print(

    "Date range:",

    dataset["report_date"].min().strftime("%Y-%m-%d"),

    "->",

    dataset["report_date"].max().strftime("%Y-%m-%d"),

)

dataset.head(3)



# ===== 生成 rolling window 切分 =====

if BUILD_ROLLING:

    splits = build_rolling_windows(

        dataset=dataset,

        train_months=TRAIN_MONTHS,

        val_months=VAL_MONTHS,

        step_months=STEP_MONTHS,

    )

    save_rolling_windows(splits, output_dir=ROLLING_OUTPUT_DIR)

    print(f"Total rolling splits: {len(splits)}")


Dataset saved: daily_text_3m_return_dataset.csv, samples=24062
Date range: 2017-01-04 -> 2025-12-05
Total rolling splits: 84


In [2]:
# ===== 新增独立Cell：日度数据集生成（不依赖上面任何Cell） =====

from generate_dataset_by_day import DayLevelDatasetBuilder, build_rolling_windows, save_rolling_windows



# 路径与参数

REPORTS_DIR = "reports_txt_by_day"

RETURNS_FILE = "all_stock_returns.csv"   # 若不存在会自动尝试 all_stock_return.csv

MAPPING_FILE = "Eastmoney_report_pdf_download/HS300.csv"

OUTPUT_FILE = "daily_text_3m_return_dataset.csv"

HORIZON_MONTHS = 3

MIN_TEXT_LENGTH = 50



# rolling window 参数

BUILD_ROLLING = True

TRAIN_MONTHS = 24

VAL_MONTHS = 3

STEP_MONTHS = 1

ROLLING_OUTPUT_DIR = "rolling_windows"



# 1) 构建主数据集

builder = DayLevelDatasetBuilder(

    reports_dir=REPORTS_DIR,

    returns_file=RETURNS_FILE,

    mapping_file=MAPPING_FILE,

    min_text_length=MIN_TEXT_LENGTH,

    horizon_months=HORIZON_MONTHS,

)



dataset = builder.build_dataset()

if dataset.empty:

    raise ValueError("No valid samples generated. Please check reports/returns/mapping paths and data coverage.")



dataset.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"Dataset saved: {OUTPUT_FILE}, samples={len(dataset)}")

print(

    "Date range:",

    dataset["report_date"].min().strftime("%Y-%m-%d"),

    "->",

    dataset["report_date"].max().strftime("%Y-%m-%d"),

)

display(dataset.head(3))



# 2) 生成 rolling window 切分

if BUILD_ROLLING:

    splits = build_rolling_windows(

        dataset=dataset,

        train_months=TRAIN_MONTHS,

        val_months=VAL_MONTHS,

        step_months=STEP_MONTHS,

    )

    save_rolling_windows(splits, output_dir=ROLLING_OUTPUT_DIR)

    print(f"Total rolling splits: {len(splits)}")


Dataset saved: daily_text_3m_return_dataset.csv, samples=24062
Date range: 2017-01-04 -> 2025-12-05


,text,label,report_date,stock_code,company_name,horizon_months,source_file
0,Acquisition of a famous American ophthalmology...,0.008552,2017-01-04,300015,爱尔眼科,3,reports_txt_by_day/20170104/爱尔眼科/20170104_东兴证券...
1,"El ophthalmology, 300015, buys American excell...",0.008552,2017-01-04,300015,爱尔眼科,3,reports_txt_by_day/20170104/爱尔眼科/20170104_西南证券...
2,"Year-round sales are higher than expected, lea...",0.103483,2017-01-04,600066,宇通客车,3,reports_txt_by_day/20170104/宇通客车/20170104_东吴证券...


Total rolling splits: 84
